In [10]:
import requests
import pandas as pd

api_base_url = "https://anapioficeandfire.com/api"

#Получение информации
def get_resource(resource, condition1=None):
  list_data = []
  count = 1
  while True:
    if condition1 is not None:
      params={"page": count, "pageSize": 40}
      params[condition1] = "true"
      data_temp = requests.get(f'{api_base_url}/{resource}', params=params)
    else:
      data_temp = requests.get(f'{api_base_url}/{resource}', params={"page": count, "pageSize": 40})

    if not data_temp:
        break
    data=data_temp.json()
    list_data.extend(data)

    if 'rel="next"' not in data_temp.headers.get("Link", ""):
      return list_data
    count+=1


#Сохранение книг в Датафрейм
df_books = pd.DataFrame(get_resource("books"))
print(f'Количество полученных книг: {len(df_books)}')
print(df_books[["name","authors","numberOfPages","released"]].head(5))

#Сохранение домов Вестероса в Датафрейм
df_houses = pd.DataFrame(get_resource("houses"))
print(f'\nКоличество домов Вестероса: {len(df_houses)}')
print(df_houses[["name","region","titles"]].head(5))

#Сохранение домов Вестероса, у которых есть девиз, в Датафрейм
df_houses_with_motto = pd.DataFrame(get_resource("houses","hasWords"))
print("\nВ параметре hasTitles из задания лежат титулы, принадлежащие дому. Девизы находятся в hadWords, поэтому использован этот параметр")
print(f'Количество домов, у которых есть девиз: {len(df_houses_with_motto)}')
print(df_houses_with_motto[["name","region","words"]].head(5))


Количество полученных книг: 12
                name                authors  numberOfPages  \
0  A Game of Thrones  [George R. R. Martin]            694   
1   A Clash of Kings  [George R. R. Martin]            768   
2  A Storm of Swords  [George R. R. Martin]            992   
3   The Hedge Knight  [George R. R. Martin]            164   
4  A Feast for Crows  [George R. R. Martin]            784   

              released  
0  1996-08-01T00:00:00  
1  1999-02-02T00:00:00  
2  2000-10-31T00:00:00  
3  2005-03-09T00:00:00  
4  2005-11-08T00:00:00  

Количество домов Вестероса: 444
                          name           region titles
0                 House Algood  The Westerlands     []
1  House Allyrion of Godsgrace            Dorne     []
2                  House Amber        The North     []
3                House Ambrose        The Reach     []
4   House Appleton of Appleton        The Reach     []

В параметре hasTitles из задания лежат титулы, принадлежащие дому. Девизы находятс

In [16]:
!pip install psycopg2-binary

import psycopg2
import pandas as pd
import time

Hostname = 'hh-pgsql-public.ebi.ac.uk'
Port = 5432
Database = 'pfmegrnargs'
User = 'reader'
Password = 'NWDMCE5xdipIjRrp'

max_attempts = 10

#Подключение к БД
for attempt in range(1, max_attempts + 1):
    try:
        conn = psycopg2.connect(host=Hostname,port=Port, dbname=Database, user=User,password=Password)
        print("Подключение успешно")
        break
    except psycopg2.OperationalError as error:
        print(f"Попытка {attempt}/{max_attempts} не удалась: {error}")
        if attempt < max_attempts:
            time.sleep(5)
else:
    raise ConnectionError("Не удалось подключиться к БД после 10 попыток")

#Получение 10 строк из таблицы rnc_database
with conn.cursor() as cursor:
    cursor.execute('SELECT * FROM  rnc_database limit 10')
    rnc_10 = cursor.fetchall()

#Сохранение в Датафрейм
df_rnc_10 = pd.DataFrame(rnc_10)
print(df_rnc_10.head(5))

#Получение значения столбцов display_name, num_sequences, num_organisms, , url для 10 строк из таблицы rnc_database
with conn.cursor() as cursor:
    cursor.execute('SELECT display_name, num_sequences, num_organisms, url FROM rnc_database limit 10')
    rnc_10_2 = cursor.fetchall()

df_rnc_10_2 = pd.DataFrame(rnc_10_2,columns=['display_name', 'num_sequences', 'num_organisms', 'url'])
print("\nЗначения столбцов из таблицы rnc_database")
print(df_rnc_10_2.head(5))

cursor.close()
conn.close()

Подключение успешно
   0                          1       2          3     4          5  6     7   \
0   5 2017-05-17 00:00:00.000000  RNACEN       VEGA    98       VEGA  N         
1  26 2017-05-01 00:00:00.000000  RNACEN    GENCODE   450    GENCODE  N         
2  25 2017-05-11 00:00:00.000000  RNACEN    ENSEMBL  1118    Ensembl  Y         
3   1 2017-05-01 00:00:00.000000  RNACEN        ENA  1161        ENA  Y         
4  59 2026-04-21 09:56:18.048915  RNACEN  CIRCATLAS  1090  circAtlas  Y  None   

          8          9       10    11        12        13      14  \
0       VEGA  PRJEB4568     NaN   NaN       NaN         0       0   
1    GENCODE              889.0  32.0  205012.0     47677       2   
2    Ensembl             1716.0  11.0  347561.0   3373550     267   
3        ENA              412.0  10.0  900074.0  12086180  814855   
4  circAtlas       None   379.0  20.0    2808.0   2800521      10   

                                                  15  \
0  is a repository for